<h3>문자, 재귀, 토큰 청킹</h3>
<p>하나의 짧은 배송·환불 규정을 세 가지 방법으로 나눕니다. 텍스트를 조각으로 나누는 각 스플리터 객체를 만든 직후 결과 본문과 길이를 확인합니다.</p>
<h3>스플리터 객체와 실제 분할 시점</h3><p>스플리터 생성자는 규칙 객체만 만들고 <code>split_text(text)</code>를 호출할 때 문자열을 나누어 청크 목록을 반환합니다. <code>chunk_size</code>는 목표 길이이며 경계가 없으면 실제 청크가 이를 넘을 수 있습니다.</p><p>문자 방식은 한 경계, 재귀 방식은 큰 경계부터 여러 경계, 토큰 방식은 토크나이저 단위를 사용합니다. 같은 <code>chunk_size</code> 숫자라도 문자 수와 토큰 수는 같지 않습니다.</p>

In [1]:
# %pip은 이 노트북의 현재 커널에 필요한 패키지를 설치합니다.
# 버전을 고정하면 학생 환경에서도 같은 클래스와 인자를 사용할 수 있습니다.
%pip install -q "langchain-text-splitters==1.1.2" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# 세 실험에 같은 원문을 사용해야 분할 기준의 차이만 비교할 수 있습니다.
text = """배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.

환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다."""

print(text)

배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.

환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.


<h3>1단계: 문자 기준으로 나누기</h3>
<p><code>CharacterTextSplitter</code>는 지정한 구분자를 기준으로 조각을 만든 뒤, <code>chunk_size</code>를 목표 크기로 삼아 조각을 합칩니다. 구분자가 없는 긴 조각은 이 크기를 넘을 수 있으므로 실제 길이를 확인합니다.</p>

In [3]:
# 공백을 경계로 사용하고, 청크의 목표 길이를 35자로 정합니다. 나눌 경계가 없는 긴 조각은 이를 넘을 수 있습니다.
character_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=35,
    chunk_overlap=5,
)
character_chunks = character_splitter.split_text(text)

for index, chunk in enumerate(character_chunks, start=1):
    print(f"{index}: {chunk!r} | 문자 수={len(chunk)}")

1: '배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더' | 문자 수=31
2: '이틀이 더 걸립니다.\n\n환불은 수령 후 7일 이내 신청합니다.' | 문자 수=34
3: '사용한 상품은 환불할 수 없습니다.' | 문자 수=19


<h3>2단계: 재귀 방식으로 나누기</h3>
<p>재귀 스플리터는 먼저 빈 줄에서 나누고, 조각이 여전히 길면 문장 끝, 그다음 공백 순서로 나눌 위치를 찾습니다. 빈 줄처럼 큰 단위의 경계에서 나눌 수 없을 때 다음 경계를 사용합니다.</p>

In [4]:
# separators의 앞쪽 항목일수록 먼저 시도합니다.
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", ". ", " ", ""],
    chunk_size=35,
    chunk_overlap=5,
    keep_separator="end",
)
recursive_chunks = recursive_splitter.split_text(text)

for index, chunk in enumerate(recursive_chunks, start=1):
    print(f"{index}: {chunk!r} | 문자 수={len(chunk)}")

1: '배송은 결제 다음 날 시작합니다.' | 문자 수=18
2: '제주 지역은 이틀이 더 걸립니다.' | 문자 수=18
3: '환불은 수령 후 7일 이내 신청합니다.' | 문자 수=21
4: '사용한 상품은 환불할 수 없습니다.' | 문자 수=19


<h3>3단계: 조건 하나만 바꾸기</h3>
<p>재귀 방식은 그대로 두고 <code>chunk_size</code>만 35에서 50으로 바꿉니다. 청크 수와 본문이 어떻게 달라지는지 확인합니다.</p>

In [5]:
# 이전 셀과 비교할 때 chunk_size만 바꿉니다.
larger_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", ". ", " ", ""],
    chunk_size=50,
    chunk_overlap=5,
    keep_separator="end",
)
larger_chunks = larger_splitter.split_text(text)

print("35자 설정:", len(recursive_chunks), "개")
print("50자 설정:", len(larger_chunks), "개")
for chunk in larger_chunks:
    print(repr(chunk), "| 문자 수=", len(chunk))

35자 설정: 4 개
50자 설정: 2 개
'배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.' | 문자 수= 37
'환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.' | 문자 수= 41


<h3>4단계: 토큰 기준으로 나누기</h3>
<p>토큰 스플리터는 모델 입력 한도를 다룰 때 사용합니다. 한국어 한 글자와 한 토큰은 같은 단위가 아니므로 토큰 수를 직접 확인합니다.</p>

In [6]:
# from_tiktoken_encoder()는 토큰 수를 길이 함수로 사용하는 재귀 스플리터를 만듭니다.
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=20,
    chunk_overlap=4,
)
token_chunks = token_splitter.split_text(text)
encoding = tiktoken.get_encoding("cl100k_base")

for index, chunk in enumerate(token_chunks, start=1):
    token_count = len(encoding.encode(chunk))
    print(f"{index}: {chunk!r} | 토큰 수={token_count}")

1: '배송은 결제 다음 날 시작합니다. 제주 지역은' | 토큰 수=20
2: '지역은 이틀이 더 걸립니다.' | 토큰 수=18
3: '환불은 수령 후 7일 이내 신청합니다.' | 토큰 수=19
4: '사용한 상품은 환불할 수 없습니다.' | 토큰 수=17


<h3>실패 확인</h3>
<p><strong>Before 원인</strong> “배송 시작 시점과 제주 추가 기간”에 답하려면 <code>결제 다음 날</code>과 <code>제주 지역은 이틀</code>이라는 근거가 함께 필요합니다. 35자 설정에서는 두 근거가 서로 다른 청크에 놓입니다.</p>

In [7]:
# 배송 시작 시점과 제주 추가 기간이 같은 청크에 있는지 확인합니다.
required_facts = ["결제 다음 날", "제주 지역은 이틀"]
before_matches = []
for chunk in recursive_chunks:
    if all(word in chunk for word in required_facts):
        before_matches.append(chunk)

print("35자 설정에서 두 근거를 함께 담은 청크:", before_matches)
try:
    assert before_matches
except AssertionError:
    print("caught: 질문에 필요한 두 근거가 청크 경계에서 나뉘었습니다.")

35자 설정에서 두 근거를 함께 담은 청크: []
caught: 질문에 필요한 두 근거가 청크 경계에서 나뉘었습니다.


<p><strong>After 수정</strong> 앞의 3단계에서 이미 실행한 50자 결과를 다시 사용합니다. 같은 재귀 방식과 5자 오버랩을 유지한 결과에서 두 배송 문장이 한 청크에 들어와 질문에 필요한 근거를 함께 읽을 수 있는지 확인합니다.</p>

In [8]:
# 앞에서 만든 larger_chunks는 50자 목표와 5자 오버랩을 사용합니다.
after_matches = []
for chunk in larger_chunks:
    if all(word in chunk for word in required_facts):
        after_matches.append(chunk)

assert len(after_matches) == 1
print("50자 설정에서 두 근거를 함께 담은 청크:")
print(after_matches[0])

50자 설정에서 두 근거를 함께 담은 청크:
배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.
